In [1]:
# ¿Puede este extracto tributario usarse para analizar ingresos por código postal sin confundir agregados estatales?

from pathlib import Path

import pandas as pd

ROOT = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "data").is_dir() and (p / "submission").is_dir()
)
SOURCE, OUTPUT = ROOT / "data/vermont.csv", ROOT / "submission/quality_report.csv"


In [2]:
# La clave analítica es código postal y tramo de ingreso; zipcode 0 es el total estatal.

tax = pd.read_csv(SOURCE)
tax.shape, tax[["zipcode", "agi_stub", "N1", "A00100"]].head()


((1476, 147),
    zipcode  agi_stub      N1   A00100
 0        0         1  111580  1314871
 1        0         2   82760  3006587
 2        0         3   46270  2851399
 3        0         4   30070  2617891
 4        0         5   39530  5472982)

In [3]:
checks = [
    (
        "required_columns",
        "completeness",
        len({"STATE", "zipcode", "agi_stub", "N1", "A00100"} - set(tax.columns)),
    ),
    ("income_group_domain", "validity", int((~tax.agi_stub.between(1, 6)).sum())),
    (
        "postal_income_key_unique",
        "uniqueness",
        int(tax[["zipcode", "agi_stub"]].duplicated().sum()),
    ),
    ("return_count_nonnegative", "validity", int((tax.N1 < 0).sum())),
    ("statewide_total_requires_filter", "scope", int(tax.zipcode.eq(0).sum())),
]
report = pd.DataFrame(checks, columns=["rule_name", "dimension", "finding_count"])
report["status"] = [
    "REVIEW" if name == "statewide_total_requires_filter" and count else "PASS"
    for name, _, count in checks
]
report.to_csv(OUTPUT, index=False)
report


,rule_name,dimension,finding_count,status
0,required_columns,completeness,0,PASS
1,income_group_domain,validity,0,PASS
2,postal_income_key_unique,uniqueness,0,PASS
3,return_count_nonnegative,validity,0,PASS
4,statewide_total_requires_filter,scope,6,REVIEW
